# Day 5: Agentic Workflows

**LLMs for Social Science** | Oxford Spring School

## The Course at a Glance

| Day | Theme | What You Learn |
|-----|-------|----------------|
| Day 1 ✓ | Foundations | How models represent meaning, process text, and generate language |
| Day 2 ✓ | From Models to Tools | Post-training (RLHF, DPO), prompting as experimental design, model evaluation |
| Day 3 ✓ | Deploying for Research | Validation, fine-tuning (BERT and LoRA), APIs and working at scale |
| Day 4 ✓ | Social Science Applications | Information extraction, RAG, LLMs as simulated agents |
| **→ Day 5** | **Agentic Workflows** | **Tool use, autonomous research assistants, production environments** |

Each day builds on the previous one. Over four days you have built a toolkit: embeddings, prompting, validation, fine-tuning, extraction, RAG. Today you learn how to **connect these pieces into autonomous workflows** where the model decides what to do, not just what to say.

This matters because:

- Many research tasks involve **multi-step reasoning**: find relevant documents, extract information, compare across sources, synthesize findings. Doing this manually is slow; doing it with a single prompt is unreliable. Agents automate the loop.
- **Tool use** turns LLMs from text-in-text-out systems into systems that can search, compute, query databases, and write files.
- Understanding agent architecture helps you evaluate when agentic tools (Claude Code, research assistants) are useful and when they are overkill.

## Today's outline

- **Section 1:** What Is an Agent? (~35 min)
- **Section 2:** Build a ReAct Agent (~45 min)
- *Break (~15 min)*
- **Section 2 (continued):** Stress-Testing and Failure Modes (~30 min)
- **Section 3:** From Notebooks to Production (~25 min, live demo)
- **Closing** (~20 min)


## Setup

In [ ]:
#@title Install dependencies and load model
!pip install -q transformers accelerate bitsandbytes torch

import torch
import json
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load Qwen2.5-7B-Instruct in 4-bit (same as Day 4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Model loaded: {model_name} (4-bit)")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


In [ ]:
#@title Generation helper

def generate_response(messages, max_new_tokens=500, temperature=0.0):
    """Generate a response from a list of chat messages."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
        )

    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


---

# Section 1: What Is an Agent?

In every session so far, you have been the decision-maker. You decided which prompt to write, which tool to use, which results to inspect. The model did what you asked, one step at a time.

An **agent** reverses this. You give the model a goal, and it decides what steps to take: which tools to call, in what order, and what to do with the results. The model becomes the planner; you become the supervisor.

## The ReAct pattern

Most agents follow a simple loop called **ReAct** (Reasoning + Acting):

```
1. THOUGHT:  The model reasons about what it needs to do next
2. ACTION:   The model calls a tool (search, compute, extract, etc.)
3. OBSERVATION: The tool returns a result
4. Repeat until the model has enough information to answer
5. ANSWER:   The model synthesizes a final response
```

This is not magic. It is a prompt format combined with a while loop. The model generates text in a structured format, your code parses it and executes the requested action, and you feed the result back. Let's build it.


## The agent's toolkit

An agent is only as useful as its tools. Each tool is a Python function with:
- A **name** the model can reference
- A **description** the model reads to decide when to use it
- An **input format** and **output format**

We will define four tools that represent a realistic (simplified) research workflow:


In [ ]:
#@title Define the agent's tools

# --- Tool 1: Search a document collection ---

DOCUMENTS = {
    "labour_economy": "Labour proposes a National Wealth Fund to invest alongside the private sector. They commit to strict fiscal rules: the current budget must move into balance and debt must be falling as a share of the economy. They plan to create jobs through green investment.",
    "conservative_economy": "The Conservatives plan to abolish the main rate of employee National Insurance by 2030, saving the average worker over 1,300 pounds per year. They promise fiscal responsibility while reducing the tax burden on workers.",
    "labour_health": "Labour pledges to cut NHS waiting times with 40,000 additional evening and weekend appointments each week. They plan to recruit thousands more GPs and shift care from hospitals to the community.",
    "conservative_health": "The Conservatives promise to build 40 new hospitals and deliver 50,000 more nurses. They plan an integrated health and social care system where nobody has to sell their home to pay for care.",
    "labour_immigration": "Labour will reform the points-based immigration system by linking visa issuance to employer training plans. They will scrap the Rwanda deportation scheme and create a Border Security Command instead.",
    "conservative_immigration": "The Conservatives will cap the overall number of visas issued each year, set by Parliament. They plan to maintain the Rwanda partnership to deter illegal Channel crossings and process asylum claims offshore.",
    "labour_climate": "Labour will establish Great British Energy, a publicly owned clean energy company. They aim to double onshore wind, triple solar power, and quadruple offshore wind by 2030.",
    "conservative_climate": "The Conservatives commit to net zero by 2050. They will invest 9.2 billion pounds in energy efficiency for homes, schools, and hospitals and establish new national parks.",
    "libdem_health": "The Liberal Democrats promise the right to see a GP within seven days. They will introduce free personal care funded by a Health and Social Care Tax, and invest in mental health parity.",
    "libdem_climate": "The Liberal Democrats target net zero by 2045, five years ahead of the current target. They will plant 60 million trees a year and generate 80 percent of electricity from renewables by 2030.",
}

def search_documents(query):
    """Search the policy document collection. Returns the 3 most relevant documents."""
    # Simple keyword matching (in production, you would use embeddings + FAISS)
    query_words = set(query.lower().split())
    scores = {}
    for doc_id, text in DOCUMENTS.items():
        doc_words = set(text.lower().split())
        overlap = len(query_words & doc_words)
        # Boost if party name or topic is in the query
        for part in doc_id.split('_'):
            if part in query.lower():
                overlap += 3
        scores[doc_id] = overlap

    ranked = sorted(scores.items(), key=lambda x: -x[1])[:3]
    results = []
    for doc_id, score in ranked:
        results.append(f"[{doc_id}]: {DOCUMENTS[doc_id]}")
    return "\n\n".join(results) if results else "No relevant documents found."


# --- Tool 2: Extract structured information ---

def extract_policy(text):
    """Extract the policy position, target group, and justification from a text passage."""
    prompt = (
        "Extract structured information from this policy text. "
        "Return a brief JSON with keys: policy, position, target_group.\n\n"
        f"Text: {text}\n\nJSON:"
    )
    messages = [{"role": "user", "content": prompt}]
    return generate_response(messages, max_new_tokens=150)


# --- Tool 3: Compare two texts ---

def compare_positions(text_a, text_b):
    """Compare two policy positions and identify key differences."""
    prompt = (
        "Compare these two policy positions. In 2-3 sentences, identify the key differences.\n\n"
        f"Position A: {text_a}\n\n"
        f"Position B: {text_b}\n\n"
        "Key differences:"
    )
    messages = [{"role": "user", "content": prompt}]
    return generate_response(messages, max_new_tokens=200)


# --- Tool 4: Simple calculator ---

def calculate(expression):
    """Evaluate a mathematical expression. Input should be a valid Python expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# Registry: the agent will use this to know what tools are available
TOOLS = {
    "search_documents": {
        "function": search_documents,
        "description": "Search the policy document collection for relevant passages. Input: a natural language query about a policy topic or party.",
    },
    "extract_policy": {
        "function": extract_policy,
        "description": "Extract structured policy information (position, target group) from a text passage. Input: the text to analyze.",
    },
    "compare_positions": {
        "function": compare_positions,
        "description": "Compare two policy positions and identify key differences. Input: two text passages separated by ' ||| '.",
    },
    "calculate": {
        "function": calculate,
        "description": "Evaluate a mathematical expression. Input: a Python math expression (e.g., '1300 * 30000000 / 1000000000').",
    },
}

print("Agent toolkit defined:")
for name, info in TOOLS.items():
    print(f"  - {name}: {info['description'][:70]}...")


### Exercise 1a: Think like an agent

Before building the agent loop, practice the reasoning yourself. Given the research question below, write out the steps you would take using only the four tools above. Use the Thought/Action format.

This is "Be the Algorithm" for agents: you do it manually first, then automate it.


In [ ]:
### Exercise 1a: Plan a multi-step research task

# Research question: "How do Labour and Conservative policies on healthcare differ?"
#
# Write your plan using the Thought / Action / Observation format.
# You don't need to actually run the tools yet, just plan the steps.
#
# Example:
#   Thought 1: I need to find Labour's healthcare policy.
#   Action 1:  search_documents("Labour healthcare policy")
#   Observation 1: [what you expect to get back]
#
#   Thought 2: ...
#   Action 2:  ...
#   etc.

# YOUR PLAN:
#
# Thought 1:
# Action 1:
# Observation 1:
#
# Thought 2:
# Action 2:
# Observation 2:
#
# Thought 3:
# Action 3:
# Observation 3:
#
# Final answer:
#


In [ ]:
#@title Solution 1a

print("""
A reasonable plan:

Thought 1: I need to find Labour's healthcare policy.
Action 1:  search_documents("Labour healthcare NHS")
Observation 1: [Labour health document about waiting times, GPs, community care]

Thought 2: Now I need the Conservative healthcare policy.
Action 2:  search_documents("Conservative healthcare NHS")
Observation 2: [Conservative health document about hospitals, nurses, social care]

Thought 3: I have both positions. Let me compare them directly.
Action 3:  compare_positions("[Labour text]" ||| "[Conservative text]")
Observation 3: [Summary of key differences]

Thought 4: I now have enough information to answer.
Final answer: [Synthesized comparison]

Key insight: the agent does not try to answer in one step. It breaks
the problem into sub-tasks, each handled by the right tool. This is
what makes agents more reliable than a single long prompt: each step
is focused and verifiable.
""")


# Section 2: Build a ReAct Agent

Now you automate the planning process you just did by hand. The agent is three components:

1. **A system prompt** that tells the model about its tools and the Thought/Action/Observation format
2. **A parser** that extracts the tool name and input from the model's response
3. **A loop** that executes tools and feeds results back until the model is done


### Exercise 2a: The agent system prompt

Write a system prompt that:
- Explains the available tools (name + description)
- Specifies the Thought / Action / Observation format
- Tells the model to use "Final Answer:" when it has enough information

The prompt must be precise: the model will follow it literally.


In [ ]:
### Exercise 2a: Write the agent system prompt

# Build the tool descriptions from the TOOLS registry
tool_descriptions = ""
for name, info in TOOLS.items():
    tool_descriptions += f"  - {name}: {info['description']}\n"

# YOUR CODE HERE: Write the system prompt
# It should include:
# 1. The agent's role
# 2. The list of available tools
# 3. The format for Thought / Action / Observation
# 4. Instructions for when to give a Final Answer

AGENT_SYSTEM_PROMPT = ""  # YOUR CODE HERE

print(AGENT_SYSTEM_PROMPT)


In [ ]:
#@title Solution 2a

tool_descriptions = ""
for name, info in TOOLS.items():
    tool_descriptions += f"  - {name}: {info['description']}\n"

AGENT_SYSTEM_PROMPT = f"""You are a research assistant that answers questions about UK party policy by using tools.

You have access to these tools:
{tool_descriptions}
To use a tool, write your response in this exact format:

Thought: [your reasoning about what to do next]
Action: [tool_name]
Action Input: [the input to the tool]

After each Action, you will receive an Observation with the tool's output. Then continue with another Thought.

When you have gathered enough information to fully answer the question, write:

Thought: I now have enough information to answer.
Final Answer: [your complete answer, citing the sources you found]

Important rules:
- Always start with a Thought before taking an Action.
- Use one tool at a time.
- Do not make up information. Only use what the tools return.
- If the tools do not return useful information, say so in your Final Answer.
"""

print(AGENT_SYSTEM_PROMPT)


### Exercise 2b: Parse and execute

Now build the machinery that makes the loop work. You need:
1. A **parser** that reads the model's output and extracts the tool name and input
2. An **executor** that calls the right function
3. The **loop** that ties it all together


In [ ]:
#@title Parser: extract Action and Action Input from model output

def parse_agent_response(response):
    """
    Parse the agent's response to extract either a tool call or a final answer.

    Returns:
        dict with either:
          {"type": "action", "tool": tool_name, "input": tool_input}
          {"type": "final_answer", "answer": answer_text}
          {"type": "error", "message": error_description}
    """
    # Check for Final Answer
    if "Final Answer:" in response:
        answer = response.split("Final Answer:")[-1].strip()
        return {"type": "final_answer", "answer": answer}

    # Check for Action
    action_match = re.search(r'Action:\s*(.+)', response)
    input_match = re.search(r'Action Input:\s*(.+)', response)

    if action_match and input_match:
        tool_name = action_match.group(1).strip()
        tool_input = input_match.group(1).strip()
        return {"type": "action", "tool": tool_name, "input": tool_input}

    return {"type": "error", "message": f"Could not parse response: {response[:200]}"}


# Test the parser
test_responses = [
    "Thought: I need to find Labour's policy.\nAction: search_documents\nAction Input: Labour healthcare",
    "Thought: I have enough info.\nFinal Answer: Labour focuses on waiting times, Conservatives on hospitals.",
    "This is not a valid agent response.",
]

for resp in test_responses:
    parsed = parse_agent_response(resp)
    print(f"  {parsed['type']}: {str(parsed)[:80]}")


In [ ]:
#@title Executor: call the right tool

def execute_tool(tool_name, tool_input):
    """Execute a tool by name with the given input."""
    if tool_name not in TOOLS:
        return f"Error: Unknown tool '{tool_name}'. Available tools: {list(TOOLS.keys())}"

    func = TOOLS[tool_name]["function"]

    # Special handling for compare_positions (needs two inputs)
    if tool_name == "compare_positions" and "|||" in tool_input:
        parts = tool_input.split("|||")
        if len(parts) == 2:
            return func(parts[0].strip(), parts[1].strip())

    return func(tool_input)


In [ ]:
### Exercise 2b: The agent loop

def run_agent(question, max_steps=6, verbose=True):
    """
    Run the ReAct agent on a question.

    Args:
        question: The research question to answer.
        max_steps: Maximum number of tool calls before forcing a stop.
        verbose: Print each step for visibility.

    Returns:
        The agent's final answer (or a timeout message).
    """
    # Initialize the conversation with the system prompt and the question
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(max_steps):
        if verbose:
            print(f"\n{'='*50}")
            print(f"STEP {step + 1}")
            print(f"{'='*50}")

        # Get the model's response
        response = generate_response(messages, max_new_tokens=400)

        if verbose:
            print(f"MODEL OUTPUT:\n{response[:500]}")

        # Parse the response
        parsed = parse_agent_response(response)

        if parsed["type"] == "final_answer":
            if verbose:
                print(f"\nFINAL ANSWER:\n{parsed['answer']}")
            return parsed["answer"]

        elif parsed["type"] == "action":
            # Execute the tool
            tool_name = parsed["tool"]
            tool_input = parsed["input"]

            if verbose:
                print(f"\nCALLING TOOL: {tool_name}({tool_input[:80]}...)")

            observation = execute_tool(tool_name, tool_input)

            if verbose:
                print(f"OBSERVATION: {observation[:300]}")

            # Append the full exchange to the conversation
            # The model's response (with Thought + Action) becomes an assistant message
            # The observation becomes a user message (simulating the environment)
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": f"Observation: {observation}"})

        elif parsed["type"] == "error":
            if verbose:
                print(f"PARSE ERROR: {parsed['message']}")
            # Give the model a chance to recover
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": (
                "Your response was not in the correct format. "
                "Please use: Thought: ... / Action: ... / Action Input: ... "
                "OR: Final Answer: ..."
            )})

    return "Agent reached maximum steps without producing a final answer."


### Exercise 2c: Run the agent

Let the agent loose on a research question. Watch it reason, act, and synthesize.


In [ ]:
### Exercise 2c: Run the agent on a research question

# Try these questions (or write your own):

question = "How do Labour and Conservative policies on healthcare differ?"
# question = "Which party has the most ambitious climate targets?"
# question = "What is the Conservative position on immigration and how much would their NI cut cost?"

answer = run_agent(question, max_steps=6, verbose=True)


In [ ]:
#@title Solution 2c: What to look for

print("Evaluating the agent's behavior:")
print()
print("GOOD SIGNS:")
print("  - The agent searches for each party separately before comparing")
print("  - It uses compare_positions to synthesize rather than making it up")
print("  - The final answer cites specific policies from the documents")
print("  - It uses calculate for numerical questions (e.g., cost estimates)")
print()
print("PROBLEMS TO WATCH FOR:")
print("  - The agent tries to answer without searching (skips to Final Answer)")
print("  - It calls the same tool repeatedly with similar queries")
print("  - The final answer includes information not found in any Observation")
print("  - It gets stuck in a loop (this is why we have max_steps)")
print()
print("-> A 7B model will sometimes struggle with the format, especially")
print("   on multi-step questions. Frontier models (GPT-4o, Claude) follow")
print("   the ReAct format much more reliably. The architecture is the same;")
print("   the model's ability to follow structured instructions is the difference.")


---

*Take a 15-minute break. When we come back: stress-testing agents and then a live demo of production tools.*

---


### Exercise 2d: Failure modes and safeguards

Agents can fail in ways that are harder to detect than simple prompting failures. You need to know what can go wrong.


In [ ]:
### Exercise 2d: Stress-test the agent

# Test 1: A question the documents cannot answer
print("TEST 1: Unanswerable question")
print("-" * 40)
answer1 = run_agent(
    "What is the Green Party's position on nuclear power?",
    max_steps=4, verbose=True
)
print()

# Test 2: A question that requires many steps
print("\nTEST 2: Complex multi-step question")
print("-" * 40)
answer2 = run_agent(
    "Compare all three parties' positions on climate, identify which has the "
    "most ambitious target, and calculate how many trees per year the LibDems "
    "would need to plant.",
    max_steps=6, verbose=True
)


In [ ]:
#@title Solution 2d: Common agent failure modes

print("Agent failure modes you should have observed:")
print()
print("1. HALLUCINATION ON MISSING DATA")
print("   When documents do not contain the answer (Green Party question),")
print("   a well-behaved agent should say 'I could not find this information.'")
print("   A poorly-behaved agent invents an answer from its training data.")
print("   -> Safeguard: the system prompt explicitly says 'do not make up information.'")
print()
print("2. TOOL MISUSE")
print("   The agent may call the wrong tool, or pass malformed input.")
print("   For compare_positions, it needs two texts separated by |||.")
print("   If it passes only one, the tool fails.")
print("   -> Safeguard: error handling in execute_tool + recovery prompt.")
print()
print("3. INFINITE LOOPS")
print("   The agent may repeat the same search, getting the same results,")
print("   without making progress toward an answer.")
print("   -> Safeguard: max_steps parameter. In production, also track")
print("      which tools have been called with which inputs and flag repeats.")
print()
print("4. CONTEXT WINDOW OVERFLOW")
print("   Each step adds messages to the conversation. After many steps,")
print("   the context fills up and the model loses track of early observations.")
print("   -> Safeguard: summarize intermediate results, prune old messages.")
print()
print("5. FORMAT DRIFT")
print("   Smaller models sometimes 'forget' the Thought/Action format mid-run")
print("   and start generating free text. Larger models are more consistent.")
print("   -> Safeguard: the error recovery prompt we built into the loop.")


### Exercise 2e: Improving the agent (stretch)

Pick one improvement and implement it:

- **Retry logic:** If a tool call fails, automatically retry with a corrected input.
- **Step summary:** After every 3 steps, add a message summarizing what the agent has learned so far. This prevents context overflow.
- **Tool selection validation:** Before executing, check if the tool name is valid and the input is non-empty.
- **Confidence check:** Before giving a Final Answer, the agent should state how confident it is and why.


In [ ]:
### Exercise 2e: Choose one improvement and implement it

# YOUR CODE HERE
# Pick one of the improvements above and modify run_agent accordingly.
# Or, write a new research question that specifically tests your improvement.

# Example: add a step counter and summary
# After step 3, inject a message: "You have used 3 of 6 steps. Summarize
# what you have found so far and plan your remaining steps."



**Section takeaway:** An agent is a loop, not magic. The model reasons about what to do (Thought), calls a tool (Action), observes the result (Observation), and repeats. The quality depends on three things: the model's ability to follow the format, the quality and coverage of the tools, and the safeguards against failure modes. In production, you would add retry logic, context management, cost tracking, and human-in-the-loop approval for high-stakes actions.

---


# Section 3: From Notebooks to Production

## The gap

The agent you just built works, but it lives inside a Colab notebook. In production, agents need:

- **A real development environment** (not a notebook): a terminal, a file system, version control
- **Persistent tool connections** (not Python functions): databases, APIs, web search, file I/O
- **Better models**: the ReAct format is fragile with 7B models; frontier models handle it reliably
- **Observability**: logging every step, tracking costs, auditing tool calls
- **Human oversight**: approval gates for high-stakes actions (sending emails, modifying databases)

Three tools bridge this gap:


## GitHub Codespaces

[Codespaces](https://github.com/features/codespaces) gives you a full cloud development environment in your browser: VS Code with a terminal, file system, Git, Python, Node.js, and any other tools you need. No local installation.

For agent development, Codespaces solves the "works on my machine" problem. You configure the environment once (in a `devcontainer.json` file), and every collaborator gets an identical setup.

**To try it yourself (if time permits):**
1. Go to [github.com/codespaces](https://github.com/codespaces)
2. Create a new Codespace from a blank template (or from your own repo)
3. In the terminal, you have a full Linux environment

Your instructor will now demo this live.


## Claude Code

[Claude Code](https://docs.anthropic.com/en/docs/claude-code) is a command-line agent that lives in your terminal. It can:
- Read and write files in your project
- Execute shell commands
- Search and navigate codebases
- Plan multi-step tasks and execute them

For researchers, Claude Code is useful for:
- Analyzing datasets: "Load this CSV, compute summary statistics, and create a visualization"
- Writing data processing scripts: "Write a Python script that cleans and merges these three datasets"
- Iterating on analysis: "The regression shows heteroskedasticity. Add robust standard errors and re-run"

**Installation** (in a Codespace or local terminal):
```
npm install -g @anthropic-ai/claude-code
```

**Usage:**
```
cd my-research-project/
claude
```

Then talk to it naturally: "Read the data in results.csv and plot the distribution of the stance variable."

Your instructor will demo this live.


## Model Context Protocol (MCP)

[MCP](https://modelcontextprotocol.io) is an open standard that lets AI models connect to external tools and data sources. Instead of defining tools as Python functions (as we did in this notebook), MCP defines a protocol so that any tool can work with any model.

Think of it as USB for AI: just as USB lets any peripheral work with any computer, MCP lets any tool work with any AI model.

**Why it matters for research:**
- Connect Claude to your university's database, your Google Drive, or your Slack workspace
- Build custom MCP servers that expose your datasets or analysis pipelines
- Share tools across your research group: one person builds the MCP server, everyone uses it

**Architecture:**
```
Your AI model <--MCP--> MCP Server <--> Your data source
                        (a small program that translates
                         between MCP protocol and your data)
```

MCP servers are lightweight programs (often just 50-100 lines of Python or TypeScript) that expose specific capabilities. The [MCP documentation](https://modelcontextprotocol.io) has quickstart guides and example servers.

Your instructor will show a brief demo of MCP in action.


### Live demo notes

*This section is for your reference during the live demo. The instructor will demonstrate:*

1. **GitHub Codespaces:** Opening a cloud development environment, navigating the terminal, installing packages
2. **Claude Code:** Installing in the Codespace, giving it a research task, watching it plan and execute
3. **MCP (brief):** How Claude Code connects to external data through MCP servers

*If you want to try Codespaces yourself after the demo, go to [github.com/codespaces](https://github.com/codespaces). You will need a GitHub account (free tier includes 60 hours/month of Codespace usage).*

*To install Claude Code, you need an [Anthropic API key](https://console.anthropic.com/). Student pricing and free credits may be available through your institution.*


---

# Closing: The Full Arc

## Five days, one toolkit

Over five days, you have built a complete understanding of how language models work and how to use them for social science research:

**Day 1: Foundations.** You built embeddings, attention, and a generation loop from scratch. You understand that LLMs are next-token predictors, and that everything else builds on this.

**Day 2: From Models to Tools.** You learned how post-training turns a base model into an assistant, and that prompting is experimental design. You saw that prompt wording alone can shift classification results by 5--15%.

**Day 3: Deploying for Research.** You validated model outputs with Cohen's kappa, fine-tuned DeBERTa and LoRA classifiers, and saw when each approach is appropriate. You have API patterns ready for when you scale up.

**Day 4: Social Science Applications.** You extracted structured information from manifesto text, built a RAG pipeline for working with large corpora, and explored whether LLMs can simulate human survey respondents.

**Day 5: Agentic Workflows.** You built a ReAct agent that autonomously plans and executes multi-step research tasks. You saw the failure modes and safeguards. You saw production tools (Codespaces, Claude Code, MCP) that take these ideas beyond the notebook.

| Day | Theme | Status |
|-----|-------|--------|
| Day 1 | Foundations | ✓ |
| Day 2 | From Models to Tools | ✓ |
| Day 3 | Deploying for Research | ✓ |
| Day 4 | Social Science Applications | ✓ |
| Day 5 | Agentic Workflows | ✓ |


## Where to go from here

**Immediate next steps:**
- Try Claude Code on your own research project (you need an Anthropic API key)
- Set up a Codespace for your research group's codebase
- Take the RAG pipeline from Day 4 and apply it to your own corpus
- Build a classification pipeline (Day 3) for your own annotation task

**Resources:**
- [Anthropic API documentation](https://docs.anthropic.com) for Claude models
- [OpenAI API documentation](https://platform.openai.com/docs) for GPT models
- [HuggingFace](https://huggingface.co) for open models, datasets, and tutorials
- [MCP documentation](https://modelcontextprotocol.io) for building tool connections
- [Claude Code documentation](https://docs.anthropic.com/en/docs/claude-code) for terminal-based agents
- [LangGraph](https://langchain-ai.github.io/langgraph/) for building more complex agent architectures

**Stay connected:**
- Course website: [llmsforsocialscience.net](https://llmsforsocialscience.net)
- All notebooks and materials will remain available after the course
- If you are interested in doing research on LLMs for social science, reach out: we are building a research lab focused on these methods

Thank you for five days of hard work. You now have the foundations to use language models rigorously in your research.
